# Ukrainian Common Voice — Download & Metadata EDA

This notebook downloads the Ukrainian (`uk`) subset of a Hugging Face-hosted Common Voice release, inspects its schema and splits, and builds file/clip-level metadata for EDA.

**Default source:** `deepdml/common_voice_26_0` (an unofficial Hugging Face mirror of Common Voice 26.0). Common Voice 27.0 was announced in September 2026, but the official current distribution is through Mozilla Data Collective; change the source below if you have downloaded a newer release. See the [Mozilla Data Collective](https://commonvoice.mozilla.org/ia/datasets) and [mirror dataset card](https://huggingface.co/datasets/deepdml/common_voice_26_0).

> Note: loading the full dataset downloads its files and may take time and disk space. This notebook places the Hugging Face cache under `/mnt/data/ukrainian-asr/hf_cache` by default. It does not train a model or modify the dataset.


## 1. Install dependencies

In [ ]:
%pip install -U datasets huggingface_hub pandas pyarrow matplotlib seaborn soundfile

## 2. Configure dataset and storage

In [ ]:
from pathlib import Path
import os, json, math, time, warnings
import pandas as pd
import numpy as np

PROJECT_DIR = Path("/mnt/data/ukrainian-asr")
CACHE_DIR = PROJECT_DIR / "hf_cache"
OUTPUT_DIR = PROJECT_DIR / "common_voice_metadata"
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Set these to another Hugging Face dataset/release if needed.
DATASET_ID = "deepdml/common_voice_26_0"
CONFIG = "uk"
HF_HOME = str(CACHE_DIR)
os.environ["HF_HOME"] = HF_HOME
os.environ["HF_DATASETS_CACHE"] = str(CACHE_DIR / "datasets")

print("Dataset:", DATASET_ID, "| config:", CONFIG)
print("Cache:", CACHE_DIR)
print("Metadata output:", OUTPUT_DIR)

## 3. Download the Ukrainian subset

If the dataset repository requires authentication or access approval, log in with `huggingface-cli login` in the environment or use `huggingface_hub.login()` after creating a token. Do not paste tokens into a notebook you plan to share.

In [ ]:
from datasets import load_dataset

# Downloads and prepares all available splits for this language config.
# If you only want to test the schema first, use the streaming cell below instead.
dataset = load_dataset(DATASET_ID, CONFIG, cache_dir=str(CACHE_DIR / "datasets"))

print(dataset)
print("\nSplits:", list(dataset.keys()))

### Optional: inspect without downloading the full dataset

In [ ]:
# Run this cell instead of the full download cell above if you want a quick schema/sample check.
# Streaming support depends on the dataset repository and format.
#
# streamed = load_dataset(DATASET_ID, CONFIG, split="train", streaming=True)
# first = next(iter(streamed))
# print(first.keys())
# print({k: (type(v).__name__ if k == "audio" else str(v)[:200]) for k, v in first.items()})

## 4. Inspect schema, sizes, and examples

In [ ]:
schema_rows = []
for split_name, split_ds in dataset.items():
    schema_rows.append({
        "split": split_name,
        "rows": len(split_ds),
        "columns": ", ".join(split_ds.column_names),
        "features": str(split_ds.features),
    })

schema_df = pd.DataFrame(schema_rows)
display(schema_df[["split", "rows", "columns"]])
display(dataset[next(iter(dataset.keys()))].features)

# Show a few records without dumping waveforms into the notebook output.
first_split = next(iter(dataset.keys()))
sample_ds = dataset[first_split]
for i in range(min(3, len(sample_ds))):
    row = sample_ds[i]
    print(f"--- {first_split} sample {i} ---")
    for key, value in row.items():
        if key == "audio":
            print("audio:", {k: (type(v).__name__ if k == "array" else v) for k, v in value.items()})
        else:
            print(f"{key}: {str(value)[:300]}")

## 5. Build row-level metadata

This creates one row per clip with transcript and contributor metadata. Audio decoding is used to estimate duration and sample rate where possible. For large datasets, this can take a while because each audio clip must be read.

Outputs are saved as CSV and Parquet. Parquet is usually preferable for subsequent EDA because it preserves types and is smaller/faster to read.

In [ ]:
def safe_text(value):
    if value is None:
        return ""
    return str(value)

def extract_clip_metadata(row, split_name, row_idx):
    out = {
        "split": split_name,
        "row_idx": row_idx,
    }
    for key, value in row.items():
        if key == "audio":
            continue
        # Keep scalar metadata; stringify unexpected nested values.
        if value is None or isinstance(value, (str, int, float, bool, np.integer, np.floating)):
            out[key] = value
        else:
            out[key] = str(value)

    audio = row.get("audio")
    duration_sec = np.nan
    sample_rate = np.nan
    audio_path = None
    if isinstance(audio, dict):
        audio_path = audio.get("path")
        sample_rate = audio.get("sampling_rate", np.nan)
        arr = audio.get("array")
        if arr is not None:
            try:
                duration_sec = len(arr) / float(sample_rate) if sample_rate else np.nan
            except Exception:
                duration_sec = np.nan

    out["audio_path"] = audio_path
    out["duration_sec"] = duration_sec
    out["sample_rate"] = sample_rate
    text = safe_text(row.get("sentence", row.get("text", "")))
    out["transcript_chars"] = len(text)
    out["transcript_words"] = len(text.split()) if text.strip() else 0
    out["transcript_empty"] = not bool(text.strip())
    return out

metadata_by_split = {}
for split_name, split_ds in dataset.items():
    records = []
    errors = 0
    for idx, row in enumerate(split_ds):
        try:
            records.append(extract_clip_metadata(row, split_name, idx))
        except Exception as exc:
            errors += 1
            records.append({"split": split_name, "row_idx": idx, "metadata_error": str(exc)})
        if (idx + 1) % 1000 == 0:
            print(f"{split_name}: processed {idx + 1:,}/{len(split_ds):,}")
    df = pd.DataFrame(records)
    metadata_by_split[split_name] = df
    safe_split = split_name.replace("/", "_")
    df.to_parquet(OUTPUT_DIR / f"{safe_split}_metadata.parquet", index=False)
    df.to_csv(OUTPUT_DIR / f"{safe_split}_metadata.csv", index=False)
    print(f"{split_name}: saved {len(df):,} rows; row errors={errors}")

print("Saved to:", OUTPUT_DIR)

## 6. Dataset-level summary

In [ ]:
summary_rows = []
for split_name, df in metadata_by_split.items():
    duration = pd.to_numeric(df.get("duration_sec", pd.Series(dtype=float)), errors="coerce")
    text_chars = pd.to_numeric(df.get("transcript_chars", pd.Series(dtype=float)), errors="coerce")
    text_words = pd.to_numeric(df.get("transcript_words", pd.Series(dtype=float)), errors="coerce")
    summary_rows.append({
        "split": split_name,
        "clips": len(df),
        "estimated_hours": duration.sum(skipna=True) / 3600,
        "duration_known_pct": duration.notna().mean() * 100 if len(df) else np.nan,
        "mean_duration_sec": duration.mean(),
        "median_duration_sec": duration.median(),
        "p95_duration_sec": duration.quantile(0.95),
        "empty_transcripts": int(df.get("transcript_empty", pd.Series(dtype=bool)).fillna(False).sum()),
        "mean_transcript_chars": text_chars.mean(),
        "mean_transcript_words": text_words.mean(),
        "unique_speakers": df["client_id"].nunique() if "client_id" in df.columns else np.nan,
    })

summary_df = pd.DataFrame(summary_rows)
display(summary_df)
summary_df.to_csv(OUTPUT_DIR / "split_summary.csv", index=False)

## 7. Inspect metadata distributions

In [ ]:
# Combine splits for high-level analysis.
all_metadata = pd.concat(metadata_by_split.values(), ignore_index=True, sort=False)

# Which columns are present?
display(pd.DataFrame({
    "column": all_metadata.columns,
    "non_null": [all_metadata[c].notna().sum() for c in all_metadata.columns],
    "null_pct": [all_metadata[c].isna().mean() * 100 for c in all_metadata.columns],
}).sort_values("null_pct", ascending=False))

# Categorical summaries, when those columns exist.
for col in ["gender", "age", "accent", "locale", "split"]:
    if col in all_metadata.columns:
        print(f"\nTop values for {col}:")
        display(all_metadata[col].fillna("MISSING").value_counts(dropna=False).head(20).rename_axis(col).reset_index(name="clips"))

# Data-quality checks
quality = {
    "rows": len(all_metadata),
    "duplicate_transcripts_count": int(all_metadata["sentence"].duplicated().sum()) if "sentence" in all_metadata.columns else None,
    "empty_transcript_count": int(all_metadata.get("transcript_empty", pd.Series(dtype=bool)).fillna(False).sum()),
    "missing_client_id_count": int(all_metadata["client_id"].isna().sum()) if "client_id" in all_metadata.columns else None,
    "missing_duration_count": int(pd.to_numeric(all_metadata.get("duration_sec", pd.Series(dtype=float)), errors="coerce").isna().sum()),
}
print(json.dumps(quality, indent=2, ensure_ascii=False))
with open(OUTPUT_DIR / "quality_checks.json", "w", encoding="utf-8") as f:
    json.dump(quality, f, ensure_ascii=False, indent=2)

## 8. Duration and transcript-length plots

These plots help detect extremely short/long clips and unusual transcript lengths. They require duration extraction in the previous section to have completed.

In [ ]:
import matplotlib.pyplot as plt

dur = pd.to_numeric(all_metadata.get("duration_sec", pd.Series(dtype=float)), errors="coerce").dropna()
if len(dur):
    plt.figure(figsize=(9, 4))
    plt.hist(dur.clip(upper=dur.quantile(0.99)), bins=50)
    plt.xlabel("Duration (seconds; clipped at 99th percentile)")
    plt.ylabel("Number of clips")
    plt.title("Common Voice clip duration distribution")
    plt.tight_layout()
    plt.show()

chars = pd.to_numeric(all_metadata.get("transcript_chars", pd.Series(dtype=float)), errors="coerce").dropna()
if len(chars):
    plt.figure(figsize=(9, 4))
    plt.hist(chars.clip(upper=chars.quantile(0.99)), bins=50)
    plt.xlabel("Transcript length (characters; clipped at 99th percentile)")
    plt.ylabel("Number of clips")
    plt.title("Transcript length distribution")
    plt.tight_layout()
    plt.show()

## 9. Save a compact report

The metadata files are saved under `common_voice_metadata/`. Keep the downloaded dataset cache separate from the metadata outputs.

For a later training experiment, do not randomly split individual clips if the same `client_id` can occur in multiple splits. Preserve the dataset's official train/validation/test split or construct a speaker-disjoint split to reduce speaker leakage.